# LFD-Net - Remote Sensing Image Dehazing

This notebook tests pretrained weights and trains LFD-Net on the SateHaze1k-thick dataset.

**Dataset**: SateHaze1k-thick  
**Model**: LFD-Net (Light Feature Dehaze Net)  
**Metrics**: PSNR, SSIM, SAM, ERGAS (validation) + comprehensive evaluation

In [1]:
#@title 1. Install Dependencies
!pip install pytorch-msssim lpips scikit-image thop opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 5.9 MB/s eta 0:00:00


In [2]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/LFDNet_Results"
!mkdir -p "/content/drive/MyDrive/LFDNet_Results/pretrained_results"
!mkdir -p "/content/drive/MyDrive/LFDNet_Results/trained_results"
!mkdir -p "/content/drive/MyDrive/LFDNet_Results/weights"

Mounted at /content/drive


In [3]:
#@title 3. Download SateHaze1k Dataset
!wget -O Haze1k.zip "https://www.dropbox.com/s/k2i3p7puuwl2g59/Haze1k.zip?dl=1"
!unzip -q Haze1k.zip -d ./Haze1k_dataset

!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/target/265.png
!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/input/265.png
!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/target/271.png
!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/input/271.png

!ls -l ./Haze1k_dataset/Haze1k/Haze1k_thick/dataset/

--2026-06-21 23:10:30--  https://www.dropbox.com/s/k2i3p7puuwl2g59/Haze1k.zip?dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.65.18, 2620:100:6022:18::a27d:4212
Connecting to www.dropbox.com (www.dropbox.com)|162.125.65.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.dropbox.com/scl/fi/wtga5ltw5vby5x7trnp0p/Haze1k.zip?rlkey=70s52w3flhtif020nx250jru3&dl=1 [following]
--2026-06-21 23:10:30--  https://www.dropbox.com/scl/fi/wtga5ltw5vby5x7trnp0p/Haze1k.zip?rlkey=70s52w3flhtif020nx250jru3&dl=1
Reusing existing connection to www.dropbox.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://uc78e8a2fa7f2d793a029e014375.dl.dropboxusercontent.com/cd/0/inline/DC1bWQyv87018anjdDohDWOGhbmnP79enh0HqZ7EqioB_8lGxToZEGS3V_RIKV5cVE5ljiMzAJuNYicboXFrBd1caOlSVdPI3alc0Akd6-k70YbR2Z5BKv-tJevwpuAGJBa11sG6uL24aBiw4EEu26D2/file?dl=1# [following]
--2026-06-21 23:10:31--  https://uc78e8a2fa7f2d793a029e014375.dl.dropboxusercontent.

In [4]:
#@title 4. Clone LFD-Net Repository
!git clone https://github.com/sumire25/LFD-Net.git
%cd LFD-Net
!ls

Cloning into 'LFD-Net'...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 20 (delta 0), reused 20 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (20/20), 4.90 MiB | 32.16 MiB/s, done.
/content/LFD-Net
evaluate.py	      LFD-Net.ipynb	  readme_images     train.py
image_data_loader.py  model.py		  README.md
infer_multi.py	      pretrained_weights  requirements.txt


In [5]:
#@title 5. Compute FLOPs and Parameters
import torch
from thop import profile
import model

lfd_net = model.LFD_Net().cuda()
lfd_net.eval()

dummy_input = torch.randn(1, 3, 480, 640).cuda()
macs, params = profile(lfd_net, inputs=(dummy_input,))

flops_g = macs / 1e9
params_m = params / 1e6

print(f"\n--- LFD-Net Complexity ---")
print(f"FLOPs: {flops_g:.4f} G")
print(f"Parameters: {params_m:.4f} M")

[INFO] Register count_relu() for <class 'torch.nn.modules.activation.LeakyReLU'>.
[INFO] Register count_convNd() for <class 'torch.nn.modules.conv.Conv2d'>.
[INFO] Register count_adap_avgpool() for <class 'torch.nn.modules.pooling.AdaptiveAvgPool2d'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.activation.ReLU'>.
[INFO] Register zero_ops() for <class 'torch.nn.modules.container.Sequential'>.

--- LFD-Net Complexity ---
FLOPs: 27.3654 G
Parameters: 0.0902 M


## Phase 1: Test Pretrained Weights

In [9]:
#@title 6. Run Inference with Pretrained Weights
!python infer_multi.py \
  -td /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/input/ \
  -sd /content/drive/MyDrive/LFDNet_Results/pretrained_results/ \
  -w pretrained_weights/SateHaze1k-TK_best.pth

In [10]:
#@title 7. Evaluate Pretrained Model
!python evaluate.py \
  --train_folder /content/drive/MyDrive/LFDNet_Results/pretrained_results/ \
  --target_folder /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/target/

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
SAM: 4.869
ERGAS: 25.928
UIQI: 0.773
QNR: 0.476
BRISQUE: 285.569
NIQE: 1.913
Histogram: 0.887
Spectral Curve Deviation: 0.962
PSNR: 17.347
SSIM: 0.773
CIEDE2000: 11.450
LPIPS: 0.220
Downloading: "https://download.pytorc

## Phase 2: Train from Scratch

In [11]:
#@title 8. Create Required Directories
!mkdir -p trained_weights
!mkdir -p training_data_captures

In [12]:
#@title 9. Train LFD-Net
!python train.py \
  -th /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/train/input/ \
  -to /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/train/target/ \
  -vh /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/valid/input/ \
  -vo /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/valid/target/ \
  -e 50 \
  -lr 0.001 \
  --val_freq 3 \
  --save_dir /content/drive/MyDrive/LFDNet_Results/weights/

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Number of Training Images: 320
Number of Validation Images: 35
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_

## Phase 3: Test Trained Weights

In [ ]:
#@title 10. Run Inference with Trained Weights
!python infer_multi.py \
  -td /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/input/ \
  -sd /content/drive/MyDrive/LFDNet_Results/trained_results/ \
  -w /content/drive/MyDrive/LFDNet_Results/weights/best.pth

In [ ]:
#@title 11. Evaluate Trained Model
!python evaluate.py \
  --train_folder /content/drive/MyDrive/LFDNet_Results/trained_results/ \
  --target_folder /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/target/